# MTDSim Replay Visualiser

**Date:** 2026-04-24  
**Branch:** `feat/replay-viz`  
**Purpose:** Walk through the replay viewer that plays back a MTDSim event log across six situational-awareness-aligned views.

Default configuration reproduces Joo Kai Tay's flagship network and finish time (`total_nodes=100`, `total_subnets=8`, `total_layers=4`, `total_database=2`, `FINISH_TIME=15000`, `seed=42`). The demo config (30 nodes, 5000 s) is kept for fast iteration.

## Views exposed

Each tab fills the viewport. A shared transport bar (play/pause/step/scrub/speed) drives every panel from the same cursor.

| Tab          | SA-Checklist items                         | Purpose                                             |
| ------------ | ------------------------------------------ | --------------------------------------------------- |
| Overview     | —                                          | network (compact) + phase tracker + timeline (compact) |
| Network      | §1.1, §1.2, §1.3, §1.5 (live halo)         | full-screen topology with MTD halos, click a host for detail |
| Attacker     | §2.1 phase tracker, §2.4 technique, §2.3   | kill-chain tracker, technique frequency, compromised list |
| Defender     | §3.1 MTD Gantt, §3.2 resource, §3.4 interrupts | three-row Gantt + interrupt log                     |
| GAP (SA L2)  | §2.6                                       | pre-rendered GAP subgraph (postMessage highlights)   |
| Timeline     | §2.2 + §3.1 + §3.2 combined                | full-screen three-row Gantt on a shared time axis   |

Sidebar on every view: run metadata (§7.3) + live metrics (sim-t, HCR, MTD/min, current phase/host/technique) + host-detail drawer.

## 1. Imports

In [ ]:
from pathlib import Path
import sys, logging, warnings

src_path = Path('../src').resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

warnings.filterwarnings('ignore')
logging.disable(logging.CRITICAL)

from mtdsim.viz.replay.config import PRIMARY, DEMO
from mtdsim.viz.replay.runner import run_canonical_sim
from mtdsim.viz.replay.log import EventLog
from mtdsim.viz.replay.app import build_app
print('loaded replay modules')

## 2. Run the Tay primary config

If the log already exists under `notebooks/gap_out/events/`, the runner short-circuits. Pass `force=True` to regenerate.

In [ ]:
events_dir = Path('gap_out/events')
events_dir.mkdir(parents=True, exist_ok=True)

primary_log_path = run_canonical_sim(PRIMARY, scheme='random', events_dir=events_dir)
print('log:', primary_log_path, primary_log_path.stat().st_size, 'bytes')

primary_log = EventLog.load(primary_log_path)
print('events:', len(primary_log.events), 't_max:', primary_log.t_max, 'hosts:', len(primary_log.topology['nodes']))
counts = primary_log.counts_at(len(primary_log.events) - 1)
print('final counts:', counts)

## 3. Static views

You can build every figure without Dash — handy for thesis screenshots (SA-Checklist §7.2 snapshot export).

In [ ]:
from mtdsim.viz.replay.panels.network import build_network_figure
from mtdsim.viz.replay.panels.timeline import build_timeline_figure
from mtdsim.viz.replay.panels.attacker import build_phase_tracker, build_technique_bar

cursor = len(primary_log.events) // 2
t_cursor = primary_log.events[cursor]['t']
print('cursor t =', t_cursor, 'event', cursor)

fig_net = build_network_figure(
    topology=primary_log.topology,
    events=primary_log.events,
    event_index=cursor,
    height=520,
)
fig_net.show()

In [ ]:
fig_time = build_timeline_figure(primary_log, sim_t=t_cursor)
fig_time.show()

In [ ]:
phase = primary_log.attacker_cursor(cursor).get('phase')
print('current phase:', phase)
build_phase_tracker(phase).show()

## 4. Launch the interactive viewer

Inline-mode Dash. Open in a browser with the URL printed below if the iframe doesn't render inline. Drag the scrubber or hit ▶ to replay.

In [ ]:
# Optional: if you have a gap.html rendered for a subgraph attacker run, pass it here.
gap_html = None  # e.g. Path('gap_out/gap.html')

app = build_app(log=primary_log, gap_html_path=gap_html)
# jupyter_dash / dash 2.x supports mode='inline'. Fall back to external if not available.
try:
    app.run(mode='inline', host='127.0.0.1', port=8051, debug=False)
except TypeError:
    print('Inline mode unavailable; running externally on http://127.0.0.1:8051')
    app.run(host='127.0.0.1', port=8051, debug=False)

## 5. Comparing schemes on the primary network

Same network, same seed — flip MTD scheme on/off to see the network view diverge. Each call writes a separate jsonl you can pass to a second `build_app` call (or just switch `--log` on the CLI).

In [ ]:
no_mtd_path = run_canonical_sim(PRIMARY, scheme='no_mtd', events_dir=events_dir)
no_mtd_log = EventLog.load(no_mtd_path)

pairs = {
    'random MTD': primary_log.counts_at(len(primary_log.events) - 1),
    'no MTD':     no_mtd_log.counts_at(len(no_mtd_log.events) - 1),
}
for k, v in pairs.items():
    print(f'{k:10s}: {v}')
print('\nRandom scheme MTD deploys:', len(primary_log.mtd_intervals))
print('No-MTD scheme MTD deploys: ', len(no_mtd_log.mtd_intervals))

## 6. Demo config (fast-boot)

The smaller 30-node config from the 2026-04-22 GAP-subgraph demo. Use this if the primary takes too long on your machine, or when iterating on panel code.

In [ ]:
demo_path = run_canonical_sim(DEMO, scheme='random', events_dir=events_dir)
demo_log = EventLog.load(demo_path)
print('demo events:', len(demo_log.events), 'hosts:', len(demo_log.topology['nodes']))

## 7. CLI equivalent

```bash
# Default: Tay primary, random MTD. Auto-runs the sim if the log is missing.
python -m mtdsim.viz.replay

# Flip scheme / config / force rerun.
python -m mtdsim.viz.replay --config primary --scheme no_mtd
python -m mtdsim.viz.replay --config demo --force-rerun

# Pass an existing log directly (bypasses auto-run).
python -m mtdsim.viz.replay --log notebooks/gap_out/events/primary_random_42.jsonl \
    --gap-html notebooks/gap_out/gap.html
```

## 8. Summary

- `mtdsim.viz.replay.runner.run_canonical_sim(PRIMARY, scheme=…)` produces an events.jsonl that the viewer replays end-to-end.
- Six tabbed views cover the SA-Checklist MUSTs that are derivable from the v1.0 event schema (1.1, 1.2, 1.3, 2.1, 2.2, 2.4, 3.1, 3.2, 3.4, 7.1, 7.3).
- Host-detail drill-down is click-driven from either network view.
- Non-replay checklist items (agent feature panel §4.x, IDS layer §5.x, cross-run evaluation §6.x) are out of scope for this branch — they need simulator-side changes (DDQN feature emission, IDS module) before a viewer can plot them.

**Known issue still open:** `TimeNetwork.is_compromised` hard-codes 0.25 instead of using `terminate_compromise_ratio` (see `docs/known_issues.md`). The viewer surfaces this effective ratio in the sidebar but does not fix it.